# Package a VCC 2026 prediction on Kaggle, in bounded memory

This notebook runs **the same implementation that is tested locally** — `src/vcc2026/packaging.py`,
driven by `scripts/48_package_prediction.py` — on a Kaggle CPU runtime. It does not retrain,
re-predict, resample, or change a single count.

**Why a remote runtime at all.** `vcc prep` 0.2.0 calls `ad.read_h5ad(path)` before it validates
anything, so it holds the whole matrix. For trial-01 (2,167,562,410 stored entries) the CLI's own
sizing model puts that peak at **~33.5 GiB**. This notebook's path streams instead, and is expected
to peak near 1 GiB — but *expected* is not *measured*, and the notebook measures it.

**What this notebook does not do.** It does not upload anything, does not submit, does not consume
submission quota, and does not publish the notebook or the dataset. Producing a valid `.vcc` is a
statement about **format**. It is not server acceptance and it is not a score.

---

## Before you run this: prepare the input dataset (you do this, not the notebook)

Create **one private Kaggle Dataset** — do not make it public — containing four files:

| File | Where it is locally |
|---|---|
| `prediction.h5ad` | `C:\Users\ferra\vcc2026-data\artifacts\q01full\prediction.h5ad` (~3.97 GiB) |
| `source_snapshot.tar.gz` | any run directory's freeze output, or generate one (see below) |
| `gene_names.csv` | `C:\Users\ferra\vcc2026-data\raw\controls\gene_names.csv` |
| `pert_counts.csv` | `C:\Users\ferra\vcc2026-data\raw\controls\pert_counts.csv` |

To produce a source snapshot on demand (a few hundred KB, source only, no data):

```powershell
.\scripts\py.cmd -c "import sys; sys.path.insert(0,'src'); from vcc2026.manifest import snapshot_source; print(snapshot_source(r'C:\Users\ferra\Desktop\source_snapshot.tar.gz')['n_files'])"
```

Upload with the Kaggle UI (**Datasets → New Dataset → Files**, visibility **Private**) or the
Kaggle CLI from your own machine. Then attach the dataset to this notebook
(**Add Input → Datasets → your dataset**), and set the accelerator to **None (CPU)**.

The upload is the only slow part (~4 GiB). The notebook re-hashes the file after transfer and
**refuses to proceed** unless the SHA-256 matches, so a truncated or corrupted upload cannot
silently become a submission.

## 1. Measure the runtime before deciding anything

Kaggle reports a very large working filesystem and a much smaller **saved-output** allowance
(19.5 GiB at the time of writing). Those are different numbers and only one of them limits what
survives the session. Transients therefore go to scratch, and only the `.vcc` goes to
`/kaggle/working`.

In [ ]:
import os, shutil, subprocess, sys, platform
from pathlib import Path

GiB = 1024 ** 3

def _mem():
    info = {}
    meminfo = Path('/proc/meminfo')
    if meminfo.exists():
        for line in meminfo.read_text().splitlines():
            if line.startswith(('MemTotal:', 'MemAvailable:')):
                key, value = line.split(':')
                info[key] = int(value.split()[0]) * 1024
    return info

mem = _mem()
print('python      :', sys.version.split()[0], '|', platform.platform())
print('cpu count   :', os.cpu_count())
print(f"RAM total   : {mem.get('MemTotal', 0) / GiB:.2f} GiB")
print(f"RAM avail   : {mem.get('MemAvailable', 0) / GiB:.2f} GiB")
for name in ('/kaggle/input', '/kaggle/working', '/kaggle/temp', '/tmp', '/'):
    path = Path(name)
    if not path.exists():
        print(f'{name:16s} (absent)')
        continue
    usage = shutil.disk_usage(path)
    writable = os.access(path, os.W_OK)
    print(f'{name:16s} free {usage.free / GiB:8.1f} GiB  writable={writable}')
print()
print('NOTE: the free-space figure above is the working filesystem, NOT the saved-output')
print('allowance (19.5 GiB). Only what is left in /kaggle/working at commit time counts')
print('against that, which is why transients below go to scratch.')
print()
print('inputs attached:')
for entry in sorted(Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []:
    print(' ', entry)
    for child in sorted(entry.iterdir())[:20]:
        print(f'    {child.name:28s} {child.stat().st_size / GiB:8.3f} GiB')

## 2. Point the notebook at the attached dataset

Edit `DATASET` below if the inventory above shows a different directory name.

In [ ]:
# --- edit this if your dataset directory is named differently -----------------
DATASET = Path('/kaggle/input/vcc2026-trial01')

PREDICTION = DATASET / 'prediction.h5ad'
SNAPSHOT   = DATASET / 'source_snapshot.tar.gz'
GENES      = DATASET / 'gene_names.csv'
PERTS      = DATASET / 'pert_counts.csv'

EXPECTED_SHA256 = '1b7e15f49104bdb9360eb22bd69574caf3dbf8ff9de3f27991e207850dc091e2'

# Scratch holds the transient payload and its compressed copy (~4 GiB each, both
# deleted before the run ends). /kaggle/working holds only the .vcc.
SCRATCH = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')
SCRATCH = SCRATCH / 'vcc-pack'
OUTPUT  = Path('/kaggle/working/vcc')
CODE    = SCRATCH / 'code'
for d in (SCRATCH, OUTPUT, CODE):
    d.mkdir(parents=True, exist_ok=True)

missing = [str(p) for p in (PREDICTION, SNAPSHOT, GENES, PERTS) if not p.exists()]
assert not missing, f'missing from the attached dataset: {missing}'
print('prediction :', PREDICTION, f'{PREDICTION.stat().st_size / GiB:.3f} GiB')
print('scratch    :', SCRATCH)
print('output     :', OUTPUT)

## 3. Verify the prediction survived the transfer

Streamed, so this costs no memory. A mismatch stops the notebook here: packaging a file that is
not the prediction would produce a perfectly valid archive of the wrong thing.

In [ ]:
import hashlib, time

started = time.perf_counter()
digest = hashlib.sha256()
with PREDICTION.open('rb') as fh:
    for chunk in iter(lambda: fh.read(1 << 22), b''):
        digest.update(chunk)
actual = digest.hexdigest()
print('sha256   :', actual)
print('expected :', EXPECTED_SHA256)
print(f'({time.perf_counter() - started:.0f}s)')
assert actual == EXPECTED_SHA256, (
    'TRANSFER CORRUPTED OR WRONG FILE — stop here and re-upload. '
    f'got {actual}'
)
print('\nOK: the file on Kaggle is byte-for-byte the prediction produced locally.')

## 4. Install the pinned CLI and unpack our source

`vcc-cli==0.2.0` is pinned because parity was established against that version: its validators are
imported and called directly, and a different version could change a check without changing ours.

Installed with `pip` (not `uv tool install`) because the packaging module **imports** `vcc.prep`;
an isolated tool environment would put it out of reach.

In [ ]:
import tarfile

print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vcc-cli==0.2.0'],
                     capture_output=True, text=True).stdout or 'pip: done')

with tarfile.open(SNAPSHOT) as tar:
    tar.extractall(CODE)
print('source files:', sum(1 for _ in CODE.rglob('*.py')), 'python modules')

sys.path.insert(0, str(CODE / 'src'))
import vcc
from vcc._version import __version__ as cli_version
from vcc2026 import packaging as pk
print('vcc-cli     :', cli_version)
print('packaging   :', pk.__file__)
assert cli_version == '0.2.0', f'parity was established against 0.2.0, got {cli_version}'

## 5. Re-run the parity tests *on this machine*

Parity passing on a Windows laptop is not parity passing here: different anndata build, different
pandas, different zstd. These are the same full-shape fixtures the local suite uses — 300 × 400 × 3
cells, the real gene and perturbation lists, every official limit enabled — so a divergence in this
environment shows up before the artifact is produced, not after.

Takes a couple of minutes.

In [ ]:
# The parity tests read the controls bundle through the project config, so point
# the data root at a directory laid out the way they expect.
data_root = SCRATCH / 'data-root'
(data_root / 'raw' / 'controls').mkdir(parents=True, exist_ok=True)
for src in (GENES, PERTS):
    shutil.copy2(src, data_root / 'raw' / 'controls' / src.name)
os.environ['VCC2026_DATA_ROOT'] = str(data_root)
os.environ['VCC2026_ARTIFACT_ROOT'] = str(SCRATCH / 'artifacts')

result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'tests.test_packaging_parity', '-v'],
    cwd=str(CODE), capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': str(CODE / 'src')},
)
print(result.stdout[-3000:])
print(result.stderr[-6000:])
assert result.returncode == 0, 'PARITY TESTS FAILED on this runtime — do not package.'
print('\nOK: this runtime accepts and rejects exactly what vcc prep does.')

## 6. Validate and package

One command, three phases: validate (streamed, every official check), package (payload → zstd →
tar), verify (official container validator, then the archive's payload compared array by array
against the input). Transients land in scratch; only the `.vcc` is written to `/kaggle/working`.

In [ ]:
cmd = [
    sys.executable, str(CODE / 'scripts' / '48_package_prediction.py'),
    '--run-id', 'kaggle01',
    '--prediction', str(PREDICTION),
    '--out', str(OUTPUT),
    '--workdir', str(SCRATCH / 'work'),
    '--genes', str(GENES),
    '--perts', str(PERTS),
    '--expect-sha256', EXPECTED_SHA256,
    '--reserve-gib', '5',
]
print(' '.join(cmd), '\n')
started = time.perf_counter()
proc = subprocess.run(cmd, cwd=str(CODE), text=True,
                      env={**os.environ, 'PYTHONPATH': str(CODE / 'src')})
print(f'\nexit {proc.returncode} after {time.perf_counter() - started:.0f}s')
assert proc.returncode == 0, 'packaging failed — read the output above; nothing was written'

## 7. What was produced, and what it cost

The report records the measured peak RSS and the measured transient disk, not an estimate of them.

In [ ]:
import json

report = json.loads((OUTPUT / 'packaging.json').read_text())
package = report.get('package', {})
verification = report.get('verification', {})
print('input sha256      :', report['input']['sha256'])
print('stored entries    : {:,}'.format(report['layout']['nnz']))
print('archive           :', package.get('output_path'))
print('archive bytes     : {:,} ({:.2f} GiB)'.format(
    package.get('archive_bytes', 0), package.get('archive_bytes', 0) / GiB))
print('archive sha256    :', verification.get('archive_sha256'))
print('meta.json         :', json.dumps(package.get('meta')))
print('container valid   :', verification.get('official_container_validator'))
print('payload vs input  :', json.dumps(verification.get('payload_vs_input'), indent=1)[:600])
print()
print('peak RSS          : {:.2f} GiB'.format((report.get('peak_rss_bytes') or 0) / GiB))
print('peak transient disk: {:.2f} GiB'.format(package.get('peak_disk_bytes', 0) / GiB))
print('total seconds     : {:.0f}'.format(report.get('total_seconds', 0)))
print()
print('official vcc prep would have peaked at ~{:.1f} GiB (vcc.sizing model)'.format(
    report.get('official_prep_peak_gib_model', 0)))
print()
print('transformations applied to the payload:')
for item in report['transformations']:
    print('  -', item)
print()
for path in sorted(OUTPUT.rglob('*')):
    print(f'{path.stat().st_size / GiB:9.3f} GiB  {path}')
print()
print('saved-output budget is 19.5 GiB; the above is what counts against it.')

## 8. What this did and did not establish

**Established, if every cell above passed:**

- the file on this runtime is byte-for-byte the prediction produced locally (SHA-256);
- this runtime accepts and rejects exactly what `vcc prep` 0.2.0 does, on full-shape fixtures;
- the prediction passes every official check, streamed;
- the archive is a valid `.vcc` by the official container validator;
- the archive's payload carries the input's matrix **bit-identically**, the same gene axis and the
  same per-cell labels, with the one documented transformation (obs index → positional).

**Not established:**

- **server acceptance.** No submission has been made. The scoring service applies its own checks,
  and only a scored submission demonstrates it accepts this artifact.
- **any statement about prediction quality.** A valid `.vcc` is a statement about format.

**Do not** run `vcc submit` from this notebook. Download the `.vcc` from `/kaggle/working` and
submit from the project machine, following `docs/SOTTOMISSIONE.md` — which is also where the
unresolved rules question about `trial-00-controls` is recorded. `trial-00` is not to be submitted
at all.